In [ ]:
import os, subprocess, sys, json, time, glob
os.environ['PYTHONUTF8']='1'
print(subprocess.run([sys.executable,'-m','pip','install','-q','flopscope>=0.12.3,<0.13','whestbench>=0.16.2,<0.17'],capture_output=True,text=True).stdout[-500:])
from importlib.metadata import version; print('flopscope', version('flopscope'), 'whestbench', version('whestbench'))
print('cpus', os.cpu_count()); print(subprocess.run(['free','-g'],capture_output=True,text=True).stdout)

In [ ]:
hits = glob.glob('/kaggle/input/**/x10_driver.py', recursive=True); print('driver at', hits)
INPUT = os.path.dirname(hits[0])
print(sorted(os.listdir(INPUT)))

In [ ]:
os.environ['HF_XET_HIGH_PERFORMANCE']='1'
t0=time.time()
r=subprocess.run(['whest','dataset','download','aicrowd/arc-whestbench-public-2026','--revision','v2-phase2','--split','mini'],capture_output=True,text=True)
print(r.stdout[-800:], r.stderr[-800:]); print('download s', time.time()-t0)

In [ ]:
env = dict(os.environ); env.update({"MODE": "dump", "DUMP_ENV": "{\"X8_D21S\": \"0.98\", \"X2_K31_MIX\": \"0.4\", \"X8_SLOC\": \"0.2\", \"X8_SSTAR\": \"0.7\"}", "START": "0", "END": "35"})
env.update({'EST': INPUT + '/e30k_x10.py', 'OUT': '/kaggle/working/x10dump', 'PYTHONWARNINGS': 'ignore', 'PYTHONUTF8': '1'})
if 'ARMS' in env:
    arms = json.loads(env['ARMS'])
    for t, e in arms.items():
        if 'X10_TAB' in e and not e['X10_TAB'].startswith('/'): e['X10_TAB'] = INPUT + '/' + e['X10_TAB']
    env['ARMS'] = json.dumps(arms)
t0 = time.time()
p = subprocess.Popen([sys.executable, INPUT + '/x10_driver.py'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if 'Warning' in line or 'warnings.warn' in line: continue
    print(line, end='', flush=True)
p.wait(); print('exit', p.returncode, 'elapsed', time.time() - t0)
print(subprocess.run(['du', '-sh', '/kaggle/working/x10dump'], capture_output=True, text=True).stdout)